# 00 - Connect this Colab GPU to VS Code

Run this notebook in the **Colab web UI** (colab.research.google.com) once per session. It exposes this Colab kernel over a public tunnel so VS Code's Jupyter extension can attach to it directly -- after that, close this tab and do all your actual work (02-07) in VS Code, executing on this same GPU.

**Before running:** `Runtime > Change runtime type > T4 GPU` in this Colab tab (kernel type/GPU can only be chosen here, not from VS Code).

**You need a free ngrok account** -- sign up at ngrok.com, then copy your authtoken from the ngrok dashboard.

In [ ]:
NGROK_AUTH_TOKEN = ""  # paste your token from https://dashboard.ngrok.com/get-started/your-authtoken
JUPYTER_TOKEN = "colab-vscode"  # any string you choose; VS Code will need this too

assert NGROK_AUTH_TOKEN, "Set NGROK_AUTH_TOKEN above first."

In [ ]:
%%capture
!pip install -q pyngrok jupyter_http_over_ws
!jupyter serverextension enable --py jupyter_http_over_ws

In [ ]:
import subprocess
import time

from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

subprocess.Popen([
    "jupyter", "notebook",
    "--NotebookApp.allow_origin=*",
    "--ip=0.0.0.0",
    "--port=8888",
    f"--NotebookApp.token={JUPYTER_TOKEN}",
    "--no-browser",
])
time.sleep(8)

tunnel = ngrok.connect(8888, "http")
print("Jupyter server URL for VS Code:")
print(f"{tunnel.public_url}/?token={JUPYTER_TOKEN}")

## Now in VS Code

1. Open the Command Palette (`Ctrl+Shift+P`) -> **"Jupyter: Specify Jupyter Server for Connections"** -> **"Existing"** -> paste the URL printed above.
2. Open any of `notebooks/02_baseline_eval.ipynb` ... `06_evaluation.ipynb` in VS Code, click the kernel picker (top right of the notebook) -> select the remote server you just added.
3. Run cells normally -- they execute on this Colab GPU, output streams back to VS Code.

**Keep this Colab browser tab open** for the whole session -- closing it kills the tunnel and the kernel. Free-tier Colab still disconnects after ~90 min idle or ~12h total, same as usual; if it disconnects, re-run this notebook and reconnect VS Code to the new URL (it changes each time).